# 02 · Una interfaz para otros

## Propósito

Convertir el pipeline del cuaderno anterior en una herramienta que pueda usar
alguien que no programa, y después sacarla del cuaderno: que sea un programa
que corre solo. Una interfaz no es decoración: es un conjunto de decisiones
sobre qué se muestra, qué se esconde y de qué se advierte.

## Qué consume

- El modelo Whisper, que volvemos a cargar acá.
- Un audio para probar: el mismo del cuaderno 01 sirve.

## Resultados de aprendizaje

- Explicar que Gradio no es otro modelo, sino una capa sobre una función de Python.
- Conectar componentes de entrada y salida a una función propia.
- Justificar qué parámetros se exponen y cuáles se dejan fijos.
- Escribir la aplicación como un archivo `.py` que corre sin Jupyter.
- Reconocer los riesgos de privacidad de publicar un enlace de transcripción.

## Celda de reanudación

Si recién llegás, se te reinició el kernel o venís de otra sesión, corré la celda de abajo
y seguís desde acá. Vuelve a cargar el modelo, que es lo único que este cuaderno necesita
del anterior.

In [ ]:
# reanudación: se corre entera, sin leerla paso a paso.
%pip -q install faster-whisper gradio

from faster_whisper import WhisperModel
import json
import os
import time
import torch

HAY_GPU = torch.cuda.is_available()

if HAY_GPU:
    NOMBRE_MODELO = "large-v3-turbo"
    DISPOSITIVO = "cuda"
    TIPO_COMPUTO = "float16"
else:
    NOMBRE_MODELO = "small"
    DISPOSITIVO = "cpu"
    TIPO_COMPUTO = "int8"

modelo = WhisperModel(
    model_size_or_path=NOMBRE_MODELO,
    device=DISPOSITIVO,
    compute_type=TIPO_COMPUTO,
)

print("Modelo", NOMBRE_MODELO, "cargado en", DISPOSITIVO)

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **interfaz** | La parte de un programa con la que interactúa una persona. No cambia lo que el programa hace: cambia qué puede pedirle |
| **componente** | Cada pieza de la interfaz: un campo de texto, un botón, un reproductor de audio |
| **evento** | Algo que hace la persona —apretar un botón— y que dispara una función de Python |
| **Gradio** | Una librería que arma interfaces web sin escribir HTML ni JavaScript |
| **enlace temporal** | Una dirección pública que Gradio crea para que otros entren a tu app. Dura horas, no es privada |
| **script** | Un archivo `.py` que corre solo, sin cuaderno y sin Jupyter |

## Una interfaz no es neutral

Hasta ahora usaste el transcriptor como una secuencia de celdas. Funciona, pero solo para
vos: otra persona tendría que abrir Jupyter, entender el orden de ejecución y editar una
variable de Python para cambiar el audio.

**La analogía.** Pensá en un lavarropas. Adentro hace algo complicado, pero afuera te
ofrece cinco programas y dos perillas. Alguien decidió que la temperatura la elegís vos y
la velocidad de centrifugado no. Esa decisión no es técnica: es una idea sobre qué te
conviene poder tocar.

Una interfaz hace lo mismo. Cada parámetro que exponés es una invitación a que alguien lo
cambie; cada uno que escondés es una decisión que tomaste por esa persona.

**Dónde vive esto.** Es exactamente lo que vas a mirar en cualquier herramienta de IA que
uses: qué te deja configurar, qué decidió por vos, y qué información te da sobre lo que
acaba de hacer. La mayoría no te dice qué modelo usó ni con qué seguridad respondió.
Nosotros sí lo vamos a mostrar, y esa también es una decisión.

## Los componentes que vamos a usar

| Componente | Tipo de Gradio | Función | Decisión de diseño |
|---|---|---|---|
| Audio de entrada | `gr.Audio` | Permite subir un audio o grabarlo desde el navegador | Se habilitan `upload` y `microphone` para mostrar dos formas de ingreso |
| Glosario | `gr.Textbox` | Permite ofrecer términos de dominio como contexto inicial | Es opcional: no debe asumirse que un prompt siempre mejora la salida |
| Botón | `gr.Button` | Inicia la transcripción | Hace visible que la inferencia es una operación solicitada y potencialmente costosa |
| Estado | `gr.Markdown` | Muestra idioma, cantidad de segmentos, tiempo y modelo | Evita presentar la salida como una caja negra sin información de proceso |
| Texto | `gr.Textbox` | Muestra una versión legible de la transcripción | Facilita copiar y revisar, pero no reemplaza al audio original |
| Descargas | `gr.File` | Devuelve TXT, SRT y JSON | Ofrece distintos formatos para distintos usos y preserva la salida estructurada |

## La tabla completa de decisiones

Esta es la tabla que conviene tener al lado mientras leés el código. Cada fila es una
decisión que alguien tomó, con lo que gana y lo que arriesga.

| Parámetro o componente | Valor inicial | Qué hace | Razón pedagógica o de diseño | Riesgo / límite |
|---|---|---|---|---|
| `audio_path` | Archivo subido o grabado | Ruta temporal al audio recibido por Gradio | Convierte una acción de interfaz en entrada para la función Python | El archivo puede ser sensible; no subir material sin permiso |
| `sources` de `gr.Audio` | `['upload', 'microphone']` | Habilita carga y grabación | Muestra que una interfaz define modos posibles de producir datos | Grabar desde navegador no equivale a consentimiento de otras personas presentes |
| `type` de `gr.Audio` | `'filepath'` | Entrega una ruta de archivo a Python | Simplifica la función de transcripción y reutiliza el pipeline existente | La ruta es temporal y propia de la sesión |
| `glosario` | Texto vacío | Recibe vocabulario de dominio opcional | Expone una decisión contextual que puede afectar el resultado | Un glosario irrelevante puede inducir errores |
| `language` | `'es'` | Fija español como idioma esperado | Reduce ambigüedad en una práctica orientada a español | Puede ser inadecuado para audio multilingüe o variedades mezcladas |
| `vad_filter` | `True` | Filtra silencios largos o tramos sin voz | Reduce ruido de proceso y posibles salidas espurias | Puede afectar tramos de voz débil o condiciones sonoras difíciles |
| `condition_on_previous_text` | `False` | Impide condicionar cada segmento con texto anterior | Reduce propagación de errores entre segmentos | En algunos audios continuos puede perder continuidad contextual |
| `beam_size` | `5` | Explora varias alternativas durante la generación | Mantiene un equilibrio entre tiempo y estabilidad de salida | Valores mayores pueden tardar más sin mejora perceptible |
| `initial_prompt` | Solo si hay glosario | Pasa el contexto textual al modelo | Permite experimentar con vocabulario técnico | No es verificación ni corrección automática |
| `NOMBRE_MODELO` | Configuración previa del cuaderno | Define qué variante de Whisper se usa | Hace visible que la interfaz depende de recursos e infraestructura | Un modelo grande puede no entrar en memoria o tardar demasiado |
| `gr.Markdown` de estado | Texto de proceso | Explica qué ocurrió durante la transcripción | Evita una interfaz que oculte modelo, idioma y demora | La información técnica no reemplaza una evaluación de calidad |
| TXT | Archivo descargable | Devuelve texto continuo | Útil para lectura, edición y procesamiento posterior | Pierde la relación visible con tiempos |
| SRT | Archivo descargable | Devuelve subtítulos temporales | Permite revisar sincronía con audio | Los tiempos y cortes dependen del modelo |
| JSON | Archivo descargable | Devuelve segmentos estructurados | Conserva inicio, fin y texto para reuso posterior | No conserva por sí mismo toda la riqueza del evento oral |
| `share=True` | Enlace público temporal | Permite abrir la app desde un enlace | Facilita demostración y pruebas desde el navegador | No es un entorno apropiado para datos sensibles |

### ✎ Para pensar

Antes de escribir una línea de código, contestá estas cinco. Las respuestas son el diseño
de la herramienta; el código después solo las ejecuta.

- ¿Qué parámetros conviene ocultar para no abrumar a una persona usuaria?
- ¿Cuáles deben quedar visibles porque alteran significativamente el resultado?
- ¿Por qué devolver TXT, SRT y JSON en vez de un único bloque de texto?
- ¿Qué advertencias de privacidad y revisión humana debe tener una interfaz de
  transcripción?
- ¿Qué cambia cuando una herramienta transforma una voz situada en un documento
  descargable?

## ⚙ Preparar el terreno

Primero Gradio y una carpeta donde dejar lo que produzca la app. Cada transcripción lleva
una marca de tiempo en el nombre: si dos personas usan la app, una no pisa el archivo de
la otra.

In [ ]:
import gradio as gr
from pathlib import Path

CARPETA_SALIDAS = Path("salidas_gradio")
CARPETA_SALIDAS.mkdir(parents=True, exist_ok=True)

print("Gradio", gr.__version__)
print("Las salidas de la app van a:", CARPETA_SALIDAS)

## Las piezas, una por una

La app va a ser una función de Python con una interfaz encima. Antes de armarla entera,
escribimos las piezas por separado: cada una hace una cosa y se puede probar sola.

Empezamos por la conversión de tiempos, la misma del cuaderno anterior. Acá la volvemos a
escribir porque este cuaderno tiene que correr solo.

In [ ]:
def tiempo_srt(segundos):
    """Convierte 83.45 en '00:01:23,450', que es lo que pide el formato SRT."""
    horas = int(segundos // 3600)
    minutos = int((segundos % 3600) // 60)
    segundos_enteros = int(segundos % 60)
    milisegundos = int((segundos - int(segundos)) * 1000)
    return f"{horas:02d}:{minutos:02d}:{segundos_enteros:02d},{milisegundos:03d}"


print("Ejemplo:", tiempo_srt(83.45))

Ahora la transcripción. Es la misma llamada del cuaderno 01, con una diferencia: el
glosario puede venir vacío.

**Acá aparece una decisión de escritura.** Se podría armar un diccionario de parámetros y
pasarlo con `**`, que es más corto. No lo hacemos: al leerlo, esa versión esconde qué
parámetros se están usando. Dos llamadas completas ocupan más lugar y se entienden de una
pasada.

In [ ]:
def pedir_transcripcion(ruta_del_audio, glosario):
    """Llama al modelo y devuelve el generador de segmentos y la info del idioma."""
    if glosario:
        return modelo.transcribe(
            audio=ruta_del_audio, language="es", vad_filter=True,
            condition_on_previous_text=False, beam_size=5,
            initial_prompt=glosario)

    return modelo.transcribe(
        audio=ruta_del_audio, language="es", vad_filter=True,
        condition_on_previous_text=False, beam_size=5)

Y el recorrido del generador, que es donde el modelo realmente trabaja. Lo mismo que
hiciste a mano en el cuaderno anterior, envuelto en un nombre.

In [ ]:
def transcribir_audio(ruta_del_audio, glosario):
    """Transcribe un audio y devuelve sus segmentos y la info del modelo."""
    generador, info = pedir_transcripcion(ruta_del_audio, glosario)

    segmentos = []
    for segmento in generador:
        texto_limpio = segmento.text.strip()
        segmentos.append({"inicio": segmento.start, "fin": segmento.end,
                          "texto": texto_limpio})
    return segmentos, info

Las tres funciones de guardado. Una por formato, cada una de cuatro o cinco líneas:
así se ve que los tres archivos salen de la misma lista de segmentos.

In [ ]:
def guardar_txt(segmentos, ruta):
    """Escribe el texto corrido de todos los segmentos."""
    textos = []
    for segmento in segmentos:
        textos.append(segmento["texto"])

    texto_completo = " ".join(textos)
    with open(ruta, "w", encoding="utf-8") as archivo:
        archivo.write(texto_completo)
    return texto_completo

El SRT es el más largo de los tres, porque el formato pide una estructura fija:
número, línea de tiempos, texto, línea en blanco.

In [ ]:
def guardar_srt(segmentos, ruta):
    """Escribe los segmentos como subtítulos numerados con sus tiempos."""
    numero_de_subtitulo = 1
    with open(ruta, "w", encoding="utf-8") as archivo:
        for segmento in segmentos:
            momento_inicial = tiempo_srt(segmento["inicio"])
            momento_final = tiempo_srt(segmento["fin"])
            archivo.write(str(numero_de_subtitulo) + "\n")
            archivo.write(momento_inicial + " --> " + momento_final + "\n")
            archivo.write(segmento["texto"] + "\n\n")
            numero_de_subtitulo = numero_de_subtitulo + 1

Y el JSON, que es el más corto porque `json.dump` hace todo el trabajo.

In [ ]:
def guardar_json(segmentos, ruta):
    """Escribe los segmentos completos, con tildes y legibles para una persona."""
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(segmentos, archivo, ensure_ascii=False, indent=2)

**¿Por qué tres funciones y no una que guarde los tres formatos?**

Porque cada una se puede probar sola. Si mañana el SRT sale con los tiempos corridos, sabés
exactamente dónde mirar, y podés correr `guardar_srt` con una lista inventada de dos
segmentos sin transcribir nada.

Una función que hiciera las tres cosas sería más corta de escribir y bastante peor de
arreglar. Es la misma razón por la que en el cuaderno 01 separamos la llamada al modelo del
recorrido del generador: **una responsabilidad por función** es lo que después te permite
encontrar el error sin leer todo.

Y el informe de estado. Esta es la pieza que más se omite en las interfaces reales,
y la que decidimos poner: qué modelo corrió, en qué idioma, con qué seguridad, cuánto
tardó.

Sin esto, la app devuelve un texto y nadie sabe de dónde salió.

In [ ]:
def armar_estado(info, cantidad_de_segmentos, segundos, uso_glosario):
    """Arma el texto en markdown que la app muestra sobre el proceso."""
    lineas = [
        "### Transcripción terminada",
        "",
        "- Idioma detectado: `" + str(info.language) + "`",
        "- Probabilidad de idioma: `" + str(round(info.language_probability, 3)) + "`",
        "- Segmentos producidos: `" + str(cantidad_de_segmentos) + "`",
        "- Tiempo de procesamiento: `" + str(round(segundos, 1)) + " segundos`",
        "- Modelo usado: `" + NOMBRE_MODELO + "`",
        "- Glosario de dominio: `" + uso_glosario + "`",
    ]
    return "\n".join(lineas)

## La función que conecta todo

Esta es la que Gradio va a llamar cuando alguien apriete el botón. Recibe lo que la persona
cargó y devuelve las cinco cosas que la interfaz muestra.

Fijate en la primera línea: **validar la entrada antes de trabajar**. Si no hay audio, la
app lo dice; no revienta con un error de Python que nadie entiende.

In [ ]:
def transcribir_desde_interfaz(ruta_del_audio, glosario):
    """Transcribe lo que llegó por la interfaz y devuelve texto, estado y archivos."""
    if ruta_del_audio is None:
        raise gr.Error("Subí o grabá un audio antes de transcribir.")

    glosario_limpio = ""
    if glosario:
        glosario_limpio = glosario.strip()

    comienzo = time.perf_counter()
    segmentos, info = transcribir_audio(ruta_del_audio, glosario_limpio)
    demora = time.perf_counter() - comienzo

    return terminar_transcripcion(segmentos, info, demora, glosario_limpio)

La separamos en dos porque lo que sigue es otra cosa: ya no transcribe, **escribe
archivos y arma la respuesta**. Son dos responsabilidades y por eso son dos funciones.

In [ ]:
def rutas_de_salida():
    """Devuelve tres rutas con una marca de tiempo, para que no se pisen entre sí."""
    marca = str(int(time.time()))
    ruta_txt = CARPETA_SALIDAS / ("transcripcion_" + marca + ".txt")
    ruta_srt = CARPETA_SALIDAS / ("transcripcion_" + marca + ".srt")
    ruta_json = CARPETA_SALIDAS / ("segmentos_" + marca + ".json")
    return ruta_txt, ruta_srt, ruta_json

Si dos personas usan la app al mismo tiempo, la marca de tiempo evita que una
sobrescriba el archivo de la otra. Es el tipo de problema que no existe en un cuaderno y
aparece apenas algo se vuelve una herramienta compartida.

In [ ]:
def terminar_transcripcion(segmentos, info, demora, glosario_limpio):
    """Guarda los tres formatos y devuelve lo que la interfaz necesita mostrar."""
    ruta_txt, ruta_srt, ruta_json = rutas_de_salida()

    texto = guardar_txt(segmentos, ruta_txt)
    guardar_srt(segmentos, ruta_srt)
    guardar_json(segmentos, ruta_json)

    uso_glosario = "no"
    if glosario_limpio:
        uso_glosario = "sí"

    estado = armar_estado(info, len(segmentos), demora, uso_glosario)
    return texto, estado, str(ruta_txt), str(ruta_srt), str(ruta_json)

## Armar la interfaz

Gradio se arma por bloques. Primero el encabezado y las entradas, después las salidas, y al
final se conecta el botón con la función.

**La advertencia va adentro de la app**, no solo en el cuaderno: quien la use puede no
haber leído nada de esto.

In [ ]:
TEXTO_DE_PORTADA = """
# Transcriptor de audio a texto

Esta interfaz usa un modelo Whisper preentrenado para convertir audio en texto.
Podés subir un archivo o grabar desde el navegador.

**Advertencia:** revisá siempre la transcripción antes de usarla. No subas material
sensible, entrevistas privadas ni audio de otras personas sin su consentimiento
explícito. Una transcripción automática no es una cita textual.
"""

print("Portada lista.")

Ahora el bloque de entradas. `gr.Blocks` es el contenedor de toda la app, y `gr.Row`
pone lo que tenga adentro uno al lado del otro.

Cada componente se guarda en una variable, porque después vamos a necesitar nombrarlos para
conectarlos con la función.

In [ ]:
with gr.Blocks(title="Transcriptor con Whisper") as aplicacion:
    gr.Markdown(TEXTO_DE_PORTADA)

    with gr.Row():
        audio_entrada = gr.Audio(
            sources=["upload", "microphone"],
            type="filepath",
            label="Audio para transcribir")

        glosario_entrada = gr.Textbox(
            label="Glosario de dominio (opcional)",
            placeholder="Ejemplo: spaCy, tokenización, corpus, TF-IDF",
            lines=6)

    boton = gr.Button("Transcribir", variant="primary")

Las salidas. Son cuatro, y el orden importa: tiene que coincidir con el orden en que
la función devuelve sus valores.

Este bloque continúa el `with` de arriba, así que las dos celdas tienen que correrse
juntas y en orden.

In [ ]:
with aplicacion:
    estado_salida = gr.Markdown("Subí o grabá un audio y presioná **Transcribir**.")

    texto_salida = gr.Textbox(
        label="Transcripción",
        lines=15,
        placeholder="La transcripción aparecerá acá.")

    with gr.Row():
        archivo_txt = gr.File(label="Descargar TXT")
        archivo_srt = gr.File(label="Descargar SRT")
        archivo_json = gr.File(label="Descargar JSON")

Y el cableado: qué función se llama, con qué entradas, y dónde van las salidas.

`inputs` y `outputs` son listas, y **el orden tiene que coincidir** con los parámetros y
los `return` de la función. Es el error más común al armar una app con Gradio.

In [ ]:
with aplicacion:
    boton.click(
        fn=transcribir_desde_interfaz,
        inputs=[audio_entrada, glosario_entrada],
        outputs=[texto_salida, estado_salida,
                 archivo_txt, archivo_srt, archivo_json])

print("Botón conectado. La app está lista para lanzarse.")

## Lanzar

Acá aparece una diferencia real entre correr esto en tu máquina y correrlo en Colab, y vale
la pena entenderla.

En tu computadora la app queda en `localhost`: **solo vos podés entrar**. En Colab no hay
un navegador del otro lado, así que Gradio crea un túnel con una dirección pública
temporal. Eso es lo que hace `share=True`.

> **Esa dirección es pública mientras está viva.** Cualquiera con el enlace entra y puede
> subir audio. No la uses para material sensible, ni la compartas fuera de la clase.

In [ ]:
# En Colab, share=True crea un enlace temporal accesible desde afuera.
aplicacion.launch(share=True)

**La celda no termina, y está bien.** Mientras la app esté viva, el kernel queda
ocupado atendiéndola: vas a ver el indicador de "ejecutando" y ninguna otra celda va a
correr. Para recuperar el cuaderno hay que interrumpir el kernel.

Eso no es un defecto: es la diferencia entre un análisis y un servicio. Un análisis empieza,
hace algo y termina. Un servidor **no termina nunca** —queda esperando a que alguien entre—
y por eso no vive cómodo adentro de un cuaderno.

Guardate esa incomodidad: es exactamente el problema que resuelve la última parte de este
cuaderno.

### ✎ Para pensar

- Probá la app sin cargar audio y apretá Transcribir. ¿Qué pasa? Ese mensaje lo escribiste
  vos, en la primera línea de `transcribir_desde_interfaz`.
- Mirá el bloque de estado después de una transcripción. ¿Qué información te da que la app
  podría haberte ocultado? ¿Se te ocurre alguna herramienta que uses que **no** te dé eso?

## Sacarlo del cuaderno

La app funciona, pero depende de un cuaderno abierto con un kernel vivo. Si cerrás Jupyter,
se termina.

Un script no necesita nada de eso: es un archivo de texto que se corre desde una terminal.
Es también el formato que entra en un repositorio de git, el que se puede versionar, y el
que mañana se puede meter en un contenedor para que corra en un servidor.

`%%writefile` toma todo el contenido de la celda y lo guarda como archivo en vez de
ejecutarlo.

In [ ]:
%%writefile app.py
# ────────────────────────────────────────────────────────────────
# app.py
# Interfaz web para transcribir audio con Whisper.
# Devuelve la transcripción en TXT, SRT y JSON.
# Uso: python app.py
# ────────────────────────────────────────────────────────────────
import json
import time
from pathlib import Path

import gradio as gr
from faster_whisper import WhisperModel

NOMBRE_MODELO = "small"
DISPOSITIVO = "cpu"
TIPO_COMPUTO = "int8"
CARPETA_SALIDAS = Path("salidas/gradio")


def main():
    CARPETA_SALIDAS.mkdir(parents=True, exist_ok=True)
    modelo = WhisperModel(
        model_size_or_path=NOMBRE_MODELO,
        device=DISPOSITIVO,
        compute_type=TIPO_COMPUTO,
    )
    aplicacion = construir_interfaz(modelo)
    aplicacion.launch()


if __name__ == "__main__":
    main()

**Ese archivo todavía no corre**, y a propósito: le falta `construir_interfaz`, más
todas las funciones que escribiste arriba.

Completarlo es el ejercicio. No lo ejecutamos desde acá porque una app de Gradio no termina
nunca —se queda esperando visitas—, y una celda que no termina deja el cuaderno trabado. Un
servidor se lanza desde una terminal, no desde un cuaderno.

```
python app.py
```

### ✎ Para pensar

- ¿Qué hace `if __name__ == "__main__"`? ¿Qué pasaría si el script no lo tuviera y otro
  archivo hiciera `import app`?
- El script ya corre sin Jupyter. ¿Qué le faltaría para que corra **en otra computadora que
  no es la tuya**, sin que nadie instale nada a mano?
- La respuesta a esa pregunta tiene nombre, y es lo que viene después en la carrera.

## ✎ Espacio de práctica

```text
Consigna

Completá app.py hasta que corra.

1. Mové a app.py las funciones que escribiste en este cuaderno: tiempo_srt,
   transcribir_audio, las tres de guardado, armar_estado y las dos de la interfaz.
2. Escribí construir_interfaz(modelo), que devuelva el objeto gr.Blocks armado.
3. Corré 'python app.py' desde una terminal y comprobá que la app levanta.
4. Anotá en markdown qué tuviste que cambiar para que las funciones sirvieran
   afuera del cuaderno. Esa lista es la diferencia entre un cuaderno y un programa.
```

## Cierre de la unidad

### Qué construiste

1. Un corpus de audio con procedencia registrada.
2. Un pipeline de ASR: audio a texto segmentado, con tiempos.
3. Tres formatos de salida, cada uno con lo que conserva y lo que pierde.
4. Una interfaz que permite usar el transcriptor sin tocar código.
5. Un programa que corre fuera del cuaderno.

### Entrega mínima

- Los cuadernos ejecutados.
- El audio usado, o su descripción si no se puede compartir.
- `transcripcion.txt`, `transcripcion.srt` y `segmentos.json`.
- Una captura de la interfaz funcionando.
- Una nota de 300 a 500 palabras con: modelo y entorno usados; tres observaciones sobre la
  calidad de la salida; un uso posterior posible del texto; y una decisión de privacidad o
  de diseño de interfaz que te haya parecido relevante.

### Las tres ideas que sostienen todo

1. **Una transcripción es una hipótesis, no un registro.** El modelo devuelve lo más
   probable, no lo que se dijo. Nunca avisa cuando se equivoca.

2. **Convertir voz en texto cambia las condiciones de circulación de lo dicho.** Lo vuelve
   buscable, copiable e indexable. Eso recae sobre personas que hablaron sin saberlo.

3. **Una interfaz decide por quien la usa.** Qué se puede tocar, qué queda fijo, qué se
   informa y qué se calla. No hay forma de que una herramienta no tome esas decisiones: solo
   hay formas de tomarlas sin darse cuenta.

### Y ahora

El texto que produjiste vuelve a ser insumo de todo lo que ya sabés hacer: frecuencias,
TF-IDF, lemas, entidades, concordancias.

Con una advertencia que ahora podés comprobar vos: el texto transcripto no se parece al
escrito. Tiene otra puntuación, repeticiones, muletillas y cortes. `es_core_news_lg` se
entrenó con prensa. Pasale esta transcripción y mirá qué hace con ella.